In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import xarray as xr
import scipy.stats as stats
import matplotlib.pyplot as plt
import tqdm as tqdm
import sklearn 

from scipy.integrate import simpson
from sklearn.preprocessing import StandardScaler

In [2]:
#load data
import pickle
with open(r"/data/keeling/a/cstraus2/Phd_research/normalized_entropies_scale1_bw51715_binn1000_10040_masked_ptot_rounded_trivariate_normadj_priorpeak_high.pkl", "rb") as f:
   Entropy_data = pickle.load(f)

MIs = pd.read_csv('MIs_scale1_bw51715_binn1000_10040_normadj_precipprior_high.csv', index_col = 0)
trivar_MI = pd.read_csv('triMI_scale1_bw51715_binn1000_10040_normadj_precipprior_high.csv',index_col = 0)

In [3]:
trivar_MI

,station,MI_label,mutual_information,H(x),C_tx,H(y),C_ty,H(z),C_tz
0,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.098983,Peak_Precipitation_rate_in_entropy,0.739025,Prior_sm_100_entropy,0.670023,Delta_sm_100_perPtot_entropy,0.857280
1,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.134528,Peak_Precipitation_rate_in_entropy,0.743332,Prior_sm_100_entropy,0.673928,Delta_sm_10_perPtot_entropy,0.786967
2,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.099911,Peak_Precipitation_rate_in_entropy,0.739137,Prior_sm_10_entropy,0.675264,Delta_sm_100_perPtot_entropy,0.857411
3,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.142246,Peak_Precipitation_rate_in_entropy,0.744267,Prior_sm_10_entropy,0.679950,Delta_sm_10_perPtot_entropy,0.787957
4,brw,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.144587,Peak_Precipitation_rate_in_entropy,0.732323,Prior_sm_100_entropy,0.723196,Delta_sm_100_perPtot_entropy,0.876261
...,...,...,...,...,...,...,...,...,...
75,stc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.141579,Peak_Precipitation_rate_in_entropy,0.766607,Prior_sm_10_entropy,0.666162,Delta_sm_10_perPtot_entropy,0.766525
76,ste,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.209288,Peak_Precipitation_rate_in_entropy,0.725010,Prior_sm_100_entropy,0.681864,Delta_sm_100_perPtot_entropy,0.881796
77,ste,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.218815,Peak_Precipitation_rate_in_entropy,0.726122,Prior_sm_100_entropy,0.682911,Delta_sm_10_perPtot_entropy,0.758569
78,ste,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.209652,Peak_Precipitation_rate_in_entropy,0.725052,Prior_sm_10_entropy,0.683716,Delta_sm_100_perPtot_entropy,0.881847


In [6]:
bbc_e

,Avg_Precipitation_rate_in_entropy,joint_entropy_Avg_Precipitation_rate_in_Delta_sm_10,joint_entropy_Avg_Precipitation_rate_in_Delta_sm_100,joint_entropy_Avg_Precipitation_rate_in_Delta_sm_100_perPtot,joint_entropy_Avg_Precipitation_rate_in_Delta_sm_10_perPtot,Peak_Precipitation_rate_in_entropy,joint_entropy_Peak_Precipitation_rate_in_Delta_sm_10,joint_entropy_Peak_Precipitation_rate_in_Delta_sm_100,joint_entropy_Peak_Precipitation_rate_in_Delta_sm_100_perPtot,joint_entropy_Peak_Precipitation_rate_in_Delta_sm_10_perPtot,...,Prior_sm_10_entropy,joint_entropy_Prior_sm_10_Delta_sm_100_perPtot,joint_entropy_Prior_sm_10_Delta_sm_10_perPtot,Prior_sm_100_entropy,joint_entropy_Prior_sm_100_Delta_sm_100_perPtot,joint_entropy_Prior_sm_100_Delta_sm_10_perPtot,Delta_sm_10_entropy,Delta_sm_10_perPtot_entropy,Delta_sm_100_entropy,Delta_sm_100_perPtot_entropy
0,7.539029,9.359857,9.168735,8.073355,8.630936,8.252744,9.645112,9.316077,8.671536,9.229437,...,9.033373,9.403576,9.875735,9.102645,9.454164,9.967639,8.718351,7.79515,8.491276,7.11434


In [4]:
bbc_e = Entropy_data['bbc']
column_label = bbc_e.columns[bbc_e.columns.str.contains('Peak_Precipitation_rate_in') & 
              ~bbc_e.columns.str.contains('^joint') & ~bbc_e.columns.str.contains('^trivariate')]
print(column_label[0])

Peak_Precipitation_rate_in_entropy


In [53]:
brw_tri = trivar_MI.loc[trivar_MI['station'] == 'brw'].copy().reset_index(drop=True)
brw_tri

,station,MI_label,mutual_information,H(x),C_tx,H(y),C_ty,H(z),C_tz
0,brw,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.147926,Peak_Precipitation_rate_in_entropy,0.734450,Prior_sm_100_entropy,0.718781,Delta_sm_100_perPtot_entropy,0.897929
1,brw,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.189616,Peak_Precipitation_rate_in_entropy,0.739430,Prior_sm_100_entropy,0.723655,Delta_sm_10_perPtot_entropy,0.755964
2,brw,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.150749,Peak_Precipitation_rate_in_entropy,0.734787,Prior_sm_10_entropy,0.703908,Delta_sm_100_perPtot_entropy,0.898341
3,brw,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.196871,Peak_Precipitation_rate_in_entropy,0.740297,Prior_sm_10_entropy,0.709186,Delta_sm_10_perPtot_entropy,0.756850


Following Goodwell and Kumar 2017 a and b:
I(x;y) = U(x;y) + R(x,z:y)
I(z;y) = U(z;y) + R(x,z;y)


R will be Rs, the Rescaled Redundancy measure
Rs = Rmin +Is*(Rmmi-Rmin)

where Is is the normalized source dependency
Is=I(x;z)/min[H(x), H(z)]

Rmmi = min(I(x;y), I(z;y))
Rmin = max[0,-II] where II is I(x;z;y) = S-R

In [7]:
def MI_partitioning(Entropies, MI, tri_MI):
    stations = list(MI.station.unique())
    Partitions = []
    for s in stations:
        print(s)
        station_entropies = Entropies[s].copy().reset_index(drop=True)
        station_MI = MI.loc[MI['station'] == s].copy().reset_index(drop=True)
        station_triMI = tri_MI.loc[tri_MI['station'] == s].copy().reset_index(drop=True)
        
        for i in range(len(station_triMI)):
            x_var_label = station_triMI.loc[i, 'H(x)']
            y_var_label = station_triMI.loc[i, 'H(y)']
            z_var_label = station_triMI.loc[i, 'H(z)']
            tri_mi_label = station_triMI.loc[i, 'MI_label']

            II = station_triMI.loc[i, 'mutual_information']
            #z is the target variable of delta sm pep Ptot
            str_p1 = f"(?:{x_var_label}.*{y_var_label}|{y_var_label}.*{x_var_label})"
            xy_MI = station_MI.loc[station_MI['MI_label'].str.contains(str_p1, regex = True), 
                                                                'mutual_information'].iloc[0]

            str_p2 = f"(?:{z_var_label}.*{y_var_label}|{y_var_label}.*{z_var_label})"
            yz_MI = station_MI.loc[station_MI['MI_label'].str.contains(str_p2,  regex = True), 
                                                                'mutual_information'].iloc[0]
            
            str_p3 = f"(?:{z_var_label}.*{x_var_label}|{x_var_label}.*{z_var_label})"
            xz_MI = station_MI.loc[station_MI['MI_label'].str.contains(str_p3, regex = True), 
                                                                'mutual_information'].iloc[0]
            
            Hx_column_name = station_entropies.columns[station_entropies.columns.str.contains(x_var_label)
                                           & ~station_entropies.columns.str.contains('^joint')
                                           & ~station_entropies.columns.str.contains('^trivariate')]
            
            Hx = station_entropies[Hx_column_name[0]].iloc[0]
            
            Hy_column_name = station_entropies.columns[station_entropies.columns.str.contains(y_var_label)
                                           & ~station_entropies.columns.str.contains('^joint')
                                           & ~station_entropies.columns.str.contains('^trivariate')]
            
            Hy = station_entropies[Hy_column_name[0]].iloc[0]

            Hz_column_name = station_entropies.columns[station_entropies.columns.str.contains(z_var_label)
                                           & ~station_entropies.columns.str.contains('^joint')
                                           & ~station_entropies.columns.str.contains('^trivariate')]
            
            Hz = station_entropies[Hz_column_name[0]].iloc[0]


            Rmmi = np.min([yz_MI, xz_MI])
            Is = xy_MI/(np.min([Hx, Hy]))

            Rmin = np.max([0, -II])
            Rs = Rmin + Is*(Rmmi - Rmin)
            
            Ux = xz_MI - Rs
            Uy = yz_MI - Rs

            Syn = II + Rs

            Rsz = Rs/Hz
            Uxz = Ux/Hz
            Uyz = Uy/Hz
            Syn_z = Syn/Hz
            # add label 
            Partitions.append({
                "Station": s,
                "MI_label": tri_mi_label,
                "Tri_MI": II,
                "MI_xy": xy_MI,
                "MI_xz": xz_MI,
                "MI_yz": yz_MI,
                "X_var": x_var_label,
                "Unique_x": Ux,
                "Y_var": y_var_label,
                "Unique_y": Uy,
                "Z_var": z_var_label,
                "Redundancy": Rs,
                "Redundancy_zratio": Rsz,
                "Unique_xz_ratio": Uxz, 
                "Unique_yz_ratio": Uyz,
                "Synergistic": Syn,
                "Synergistic_ratio": Syn_z

            })
            

    return pd.DataFrame(Partitions)

In [8]:
Partition_values = MI_partitioning(Entropy_data, MIs, trivar_MI)

bbc
brw
bvl
cmi
dek
dxs
fai
fre
frm
icc
llc
mon
oln
orr
rnd
sfm
siu
sni
stc
ste


In [9]:
Partition_values

,Station,MI_label,Tri_MI,MI_xy,MI_xz,MI_yz,X_var,Unique_x,Y_var,Unique_y,Z_var,Redundancy,Redundancy_zratio,Unique_xz_ratio,Unique_yz_ratio,Synergistic,Synergistic_ratio
0,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.098983,6.915796,6.695547,6.762821,Peak_Precipitation_rate_in_entropy,1.084681,Prior_sm_100_entropy,1.151955,Delta_sm_100_perPtot_entropy,5.610866,0.788670,0.152464,0.161920,11.709849,1.645950
1,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.134528,6.915796,6.818457,6.930156,Peak_Precipitation_rate_in_entropy,1.104593,Prior_sm_100_entropy,1.216292,Delta_sm_10_perPtot_entropy,5.713864,0.733002,0.141703,0.156032,11.848392,1.519970
2,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.099911,6.908136,6.695547,6.744137,Peak_Precipitation_rate_in_entropy,1.090896,Prior_sm_10_entropy,1.139486,Delta_sm_100_perPtot_entropy,5.604651,0.787796,0.153338,0.160167,11.704563,1.645207
3,bbc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.142246,6.908136,6.818457,6.952787,Peak_Precipitation_rate_in_entropy,1.110922,Prior_sm_10_entropy,1.245252,Delta_sm_10_perPtot_entropy,5.707535,0.732191,0.142514,0.159747,11.849781,1.520148
4,brw,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.144587,6.902768,6.740381,6.754500,Peak_Precipitation_rate_in_entropy,1.195171,Prior_sm_100_entropy,1.209291,Delta_sm_100_perPtot_entropy,5.545210,0.790786,0.170440,0.172453,11.689796,1.667047
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,stc,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.141579,6.909718,6.853852,6.956888,Peak_Precipitation_rate_in_entropy,0.942484,Prior_sm_10_entropy,1.045520,Delta_sm_10_perPtot_entropy,5.911368,0.737793,0.117631,0.130490,12.052947,1.504318
76,ste,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.209288,6.983392,6.771588,6.838078,Peak_Precipitation_rate_in_entropy,1.250063,Prior_sm_100_entropy,1.316554,Delta_sm_100_perPtot_entropy,5.521524,0.784125,0.177524,0.186967,11.730812,1.665920
77,ste,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.218815,6.983392,6.931623,7.006519,Peak_Precipitation_rate_in_entropy,1.279606,Prior_sm_100_entropy,1.354502,Delta_sm_10_perPtot_entropy,5.652017,0.689431,0.156086,0.165222,11.870831,1.448000
78,ste,trivariate_MI_Peak_Precipitation_rate_in_Prior...,6.209652,6.976621,6.771588,6.818083,Peak_Precipitation_rate_in_entropy,1.255417,Prior_sm_10_entropy,1.301912,Delta_sm_100_perPtot_entropy,5.516171,0.783365,0.178285,0.184888,11.725823,1.665212


In [13]:

numeric_cols = Partition_values.select_dtypes(include="number").columns.tolist()
string_cols = [c for c in Partition_values.columns if c not in numeric_cols and c != "MI_label"]

agg_dict = {col: ["mean", "min", "max"] for col in numeric_cols}
for col in string_cols:
    agg_dict[col] = "first"   # just keep one representative string


agg_df = (
    Partition_values.groupby("MI_label", as_index=False)
      .agg(agg_dict)
)

agg_df.columns = [
    col if isinstance(col, str) else f"{col[0]}_{col[1]}"
    for col in agg_df.columns.to_flat_index()
]

['Station', 'X_var', 'Y_var', 'Z_var']

In [27]:
agg_df.iloc[:,22:]

,Redundancy_zratio_mean,Redundancy_zratio_min,Redundancy_zratio_max,Unique_xz_ratio_mean,Unique_xz_ratio_min,Unique_xz_ratio_max,Unique_yz_ratio_mean,Unique_yz_ratio_min,Unique_yz_ratio_max,Synergistic_mean,Synergistic_min,Synergistic_max,Synergistic_ratio_mean,Synergistic_ratio_min,Synergistic_ratio_max,Station_first,X_var_first,Y_var_first,Z_var_first
0,0.782577,0.760142,0.833948,0.154572,0.126897,0.179785,0.164472,0.130266,0.188129,11.861230,11.655711,12.793550,1.637727,1.588293,1.727842,bbc,Peak_Precipitation_rate_in_entropy,Prior_sm_100_entropy,Delta_sm_100_perPtot_entropy
1,0.710851,0.673988,0.773801,0.140666,0.118075,0.159393,0.148631,0.129624,0.167759,12.003244,11.847037,12.829913,1.475862,1.407191,1.608132,bbc,Peak_Precipitation_rate_in_entropy,Prior_sm_100_entropy,Delta_sm_10_perPtot_entropy
2,0.782991,0.760642,0.834790,0.154158,0.126420,0.180701,0.163940,0.132637,0.188316,11.867603,11.645676,12.761321,1.638608,1.592533,1.729824,bbc,Peak_Precipitation_rate_in_entropy,Prior_sm_10_entropy,Delta_sm_100_perPtot_entropy
3,0.711333,0.675766,0.770139,0.140184,0.117631,0.160205,0.149281,0.129966,0.167369,12.002842,11.824798,12.782035,1.475804,1.408948,1.602130,bbc,Peak_Precipitation_rate_in_entropy,Prior_sm_10_entropy,Delta_sm_10_perPtot_entropy
